# Fine-tune Qwen3-8B with QLoRA for the KQML agent (Google Colab)

This notebook teaches Qwen3-8B the agent's three LLM tasks (**classify**, **extract**, **SQL**) and measures it **before and after** training on the validation questions.

Repository: [github.com/ishancoderr/kqml-geo-qwen3-qlora](https://github.com/ishancoderr/kqml-geo-qwen3-qlora) — Step 3 downloads the training data from it, so there is nothing to upload.

**Before you start:** *Runtime → Change runtime type → **T4 GPU***. Then run the steps in order.

**Where results go** (set in Step 3):
- `USE_DRIVE = True` *(recommended)* — checkpoints, adapter and predictions are saved to `MyDrive/kqml_finetune/`. If Colab disconnects, run Steps 1–8 again, then Step 10 continues from the last saved epoch.
- `USE_DRIVE = False` — everything stays on Colab's own disk, which is wiped when the session ends. Step 15 downloads the adapter and predictions as a zip. A disconnect during training means starting over.

## Step 1 — Check the GPU
Confirms a GPU is attached and which one. On a T4 you should see **Tesla T4, compute capability 7.5**. The T4 has no real bfloat16 support, so this notebook will use float16 on it (bfloat16 only on newer GPUs such as A100/L4).

In [3]:
!nvidia-smi -L
import torch
print("CUDA available:", torch.cuda.is_available())
major, minor = torch.cuda.get_device_capability()
print(f"GPU: {torch.cuda.get_device_name(0)}  |  compute capability {major}.{minor}")
print("Precision used:", "bfloat16" if major >= 8 else "float16 (this GPU has no real bfloat16)")

GPU 0: Tesla T4 (UUID: GPU-1a4af646-9083-d723-3c36-925a3208215b)
CUDA available: True
GPU: Tesla T4  |  compute capability 7.5
Precision used: float16 (this GPU has no real bfloat16)


## Step 2 — Install the libraries
- **transformers** — loads Qwen3 and runs training. Pinned to 4.x (4.56 or newer) because this code uses the 4.x training API.
- **peft** — adds the small LoRA adapter (the part that is trained).
- **bitsandbytes** — loads the model in 4-bit (the "Q" in QLoRA).
- **accelerate**, **datasets** — GPU placement and the data container the trainer expects.

Takes 1–2 minutes. If Colab asks you to **restart the session**, do it, then continue with Step 3.

In [4]:
!pip install -q "transformers>=4.56,<5.0" "peft>=0.13" "bitsandbytes>=0.43" "accelerate>=0.34" datasets
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

transformers 4.57.6 | peft 0.21.0 | bitsandbytes 0.50.2


## Step 3 — Get the data and choose where results go
Downloads this repository from GitHub (the training data is in its `data/` folder) and sets where outputs are written:
- **`USE_DRIVE = True`** — mounts your Google Drive (a pop-up asks for permission); outputs go to `MyDrive/kqml_finetune/` and survive a disconnect.
- **`USE_DRIVE = False`** — outputs stay on Colab's disk and are lost when the session ends (download them in Step 15).

You should see **found** twice.

In [7]:
from pathlib import Path

USE_DRIVE = True                     # False = no Drive; download the results at the end (Step 15)
REPO      = "https://github.com/ishancoderr/kqml-geo-qwen3-qlora.git"
BRANCH    = "main"

!rm -rf /content/kqml-geo-qwen3-qlora
!git clone -q --depth 1 --branch {BRANCH} {REPO} /content/kqml-geo-qwen3-qlora
DATA_DIR   = Path("/content/kqml-geo-qwen3-qlora/data")
TRAIN_FILE = DATA_DIR / "questions_v1_train.jsonl"
VAL_FILE   = DATA_DIR / "questions_v1_val.jsonl"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/kqml_finetune")
else:
    WORK_DIR = Path("/content/kqml_finetune")             # wiped when the session ends

ADAPTER_DIR    = WORK_DIR / "adapters" / "qwen3-8b-v1"      # the trained adapter
CHECKPOINT_DIR = WORK_DIR / "checkpoints" / "qwen3-8b-v1"   # one checkpoint per epoch
RESULTS_DIR    = WORK_DIR / "results"                       # predictions, base and tuned

for d in (ADAPTER_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
for f in (TRAIN_FILE, VAL_FILE):
    print(("found    " if f.exists() else "MISSING  ") + str(f))
print("outputs go to:", WORK_DIR)

Mounted at /content/drive
found    /content/kqml-geo-qwen3-qlora/data/questions_v1_train.jsonl
found    /content/kqml-geo-qwen3-qlora/data/questions_v1_val.jsonl
outputs go to: /content/drive/MyDrive/kqml_finetune


## Step 4 — Settings
All the numbers in one place:
- **MAX_LENGTH 2048** — longest example allowed (ours is about 1,700 tokens, so none is skipped).
- **EPOCHS 3** — passes over the training data; with only ~375 examples more mostly memorises. The best epoch (lowest validation loss) is kept.
- **LEARNING_RATE 2e-4**, **LORA_RANK 16** — standard QLoRA values.
- **BATCH_SIZE 1 × GRAD_ACCUM 8** — the T4 holds one example at a time; gradients from 8 are added up before each update (an effective batch of 8).
- **MAX_NEW_TOKENS** — how long an answer may be when testing, per task.

In [8]:
import torch

MODEL_NAME    = "Qwen/Qwen3-8B"
MAX_LENGTH    = 2048
EPOCHS        = 3
LEARNING_RATE = 2e-4
LORA_RANK     = 16
BATCH_SIZE    = 1
GRAD_ACCUM    = 8
SEED          = 42
MAX_NEW_TOKENS = {"classify": 32, "extract": 384, "sql": 512}

DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("compute dtype:", DTYPE)

compute dtype: torch.float16


## Step 5 — Read the data
Each line of the JSONL is **one LLM call** of the agent: a system prompt, a user message and the correct assistant answer, plus a `meta` block. From `meta` we keep the **task** (classify / extract / sql) so the scores can be shown per task. You should see 375 training and 38 validation examples.

In [9]:
import json
from collections import Counter

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            msgs = rec["messages"]
            assert [m["role"] for m in msgs] == ["system", "user", "assistant"], msgs
            meta = rec.get("meta", {})
            rows.append({"task": meta.get("task", "all"), "qid": meta.get("qid", ""),
                         "prompt_messages": msgs[:2], "answer": msgs[2]["content"]})
    return rows

train_rows = load_jsonl(TRAIN_FILE)
val_rows   = load_jsonl(VAL_FILE)
print("train:", len(train_rows), dict(Counter(r["task"] for r in train_rows)))
print("val:  ", len(val_rows),   dict(Counter(r["task"] for r in val_rows)))

train: 375 {'classify': 109, 'extract': 100, 'sql': 166}
val:   38 {'classify': 11, 'extract': 10, 'sql': 17}


## Step 6 — Tokenizer: turn each example into training data
How one example becomes training data:
1. The system + user messages are rendered with **Qwen3's own chat template, thinking turned off** (`enable_thinking=False`). That adds an empty `<think></think>` block — exactly the prompt the model gets when the agent uses it.
2. The **target** is the correct answer followed by `<|im_end|>`, so the model also learns where to stop.
3. Every prompt token gets the label `-100`, which means **"no loss here"** — the model learns to produce the answer, not to repeat the long prompt.
4. An example longer than MAX_LENGTH is **skipped, never cut short** (cutting would remove the answer).

The output shows token lengths per task, the end of one prompt, and the exact text the loss is computed on.

In [10]:
from transformers import AutoTokenizer

END = "<|im_end|>"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
END_ID = tokenizer.convert_tokens_to_ids(END)

def prompt_text(row):
    return tokenizer.apply_chat_template(row["prompt_messages"], tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)

def encode(row):
    prompt_ids = tokenizer(prompt_text(row), add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(row["answer"] + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + answer_ids,
            "labels": [-100] * len(prompt_ids) + answer_ids}

def encode_all(rows, name):
    kept, skipped, lengths = [], Counter(), {}
    for r in rows:
        item = encode(r)
        lengths.setdefault(r["task"], []).append(len(item["input_ids"]))
        if len(item["input_ids"]) > MAX_LENGTH:
            skipped[r["task"]] += 1
            continue
        kept.append(item)
    print(f"{name}: {len(kept)}/{len(rows)} kept")
    for t, ls in sorted(lengths.items()):
        print(f"   {t:<9} n={len(ls):<4} tokens mean {sum(ls) // len(ls):<5} max {max(ls):<5} skipped {skipped[t]}")
    return kept

train_data = encode_all(train_rows, "train")
val_data   = encode_all(val_rows, "val")

sample = val_rows[0]
item = encode(sample)
print("\n--- end of the prompt the model sees ---\n..." + prompt_text(sample)[-200:])
print("--- the only part the loss is computed on ---")
print(tokenizer.decode([t for t, l in zip(item["input_ids"], item["labels"]) if l != -100]))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

train: 375/375 kept
   classify  n=109  tokens mean 1284  max 1304  skipped 0
   extract   n=100  tokens mean 884   max 1456  skipped 0
   sql       n=166  tokens mean 1110  max 1692  skipped 0
val: 38/38 kept
   classify  n=11   tokens mean 1283  max 1294  skipped 0
   extract   n=10   tokens mean 881   max 1420  skipped 0
   sql       n=17   tokens mean 1152  max 1685  skipped 0

--- end of the prompt the model sees ---
...ATED"}

"What is the population of Paris?"
→ {"query_type":"UNRELATED"}
<|im_end|>
<|im_start|>user
How many residents did Brandenburg have in 2020?<|im_end|>
<|im_start|>assistant
<think>

</think>


--- the only part the loss is computed on ---
{"query_type": "DIRECT_LOOKUP"}<|im_end|>


## Step 7 — Load Qwen3-8B in 4-bit
Downloads the model (about **16 GB**, a few minutes, once per Colab session) and loads it **quantised to 4-bit NF4** — the "Q" in QLoRA. In 4-bit the 8-billion-parameter model needs only about 6 GB of GPU memory, which is what makes training on a 16 GB T4 possible.

The generation settings are switched to plain greedy decoding (no sampling), so every test answer is deterministic.

If the session **crashes while loading**, it ran out of *system RAM* (free Colab has about 12 GB), not GPU memory.

In [11]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    ),
    device_map={"": 0},
    dtype=DTYPE,
)
gen = model.generation_config
gen.do_sample, gen.temperature, gen.top_p, gen.top_k = False, None, None, None
print(f"loaded — GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded — GPU memory in use: 6.1 GB


## Step 8 — Testing helpers
Defines how the model is tested on the validation examples (nothing runs yet):
- The model answers each example with greedy decoding.
- **classify / extract:** the answer must be valid JSON and **exactly equal** to the correct JSON.
- **sql:** the SQL statement is compared after ignoring spacing and letter case.
- **valid JSON** is reported separately — it shows whether the model at least kept the output format.

Every prediction is saved to Drive next to the correct answer. Note: the SQL check is strict — a correct statement written differently counts as wrong. Running the SQL against the database is the real check, later.

In [13]:
import re, time, statistics

def clean(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    fenced = re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", text, flags=re.S)
    return fenced.group(1).strip() if fenced else text

def parse(text):
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None

def norm_sql(sql):
    return re.sub(r"\s+", " ", sql.strip().rstrip(";")).lower()

def score(task, gold, pred):
    p, g = parse(pred), parse(gold)
    if p is None:
        return False, False                      # not even valid JSON
    if task == "sql":
        ok = isinstance(p, dict) and isinstance(p.get("sql"), str) and norm_sql(p["sql"]) == norm_sql(g["sql"])
        return True, ok
    return True, p == g

@torch.no_grad()
def answer(row):
    inputs = tokenizer(prompt_text(row), return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS.get(row["task"], 512), do_sample=False,
                         eos_token_id=END_ID, pad_token_id=tokenizer.pad_token_id)
    return clean(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

def print_table(tag, results):
    print(f"\n{tag}\n  {'task':<9}{'n':>4}{'valid JSON':>12}{'correct':>10}{'s/answer':>10}")
    for task in ["classify", "extract", "sql", "TOTAL"]:
        g = results if task == "TOTAL" else [r for r in results if r["task"] == task]
        if g:
            print(f"  {task:<9}{len(g):>4}{sum(r['valid_json'] for r in g) / len(g):>12.0%}"
                  f"{sum(r['correct'] for r in g) / len(g):>10.0%}{statistics.mean(r['seconds'] for r in g):>10.1f}")

def run_eval(tag):
    model.eval()
    model.config.use_cache = True
    results = []
    for i, row in enumerate(val_rows, 1):
        t0 = time.perf_counter()
        pred = answer(row)
        secs = time.perf_counter() - t0
        valid, correct = score(row["task"], row["answer"], pred)
        results.append({"qid": row["qid"], "task": row["task"], "gold": row["answer"], "pred": pred,
                        "valid_json": valid, "correct": correct, "seconds": round(secs, 2)})
        print(f"[{i}/{len(val_rows)}] {row['qid']:<4} {row['task']:<8} {'ok ' if correct else 'BAD'} {secs:5.1f}s")
    path = RESULTS_DIR / f"eval_{tag}.jsonl"
    path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in results) + "\n", encoding="utf-8")
    print_table(tag, results)
    print("saved:", path)
    return results

print("helpers ready")

helpers ready


## Step 9 — BEFORE training: test the base model
Scores plain Qwen3-8B (no training) on the 38 validation examples. This is your **"Qwen3 8B, not fine-tuned"** row in the thesis comparison. Takes roughly 5–10 minutes.

In [14]:
base_results = run_eval("base")

[1/38] d10  classify ok    4.0s
[2/38] d10  extract  ok    5.3s
[3/38] d10  sql      ok    7.7s
[4/38] d20  classify ok    2.6s
[5/38] d20  extract  ok    8.2s
[6/38] d20  sql      ok    8.7s
[7/38] d20  sql      ok    7.9s
[8/38] d20  sql      ok    8.7s
[9/38] g10  classify ok    2.4s
[10/38] g10  extract  ok    2.3s
[11/38] g20  classify ok    2.6s
[12/38] g20  extract  ok    2.9s
[13/38] o10  classify ok    2.6s
[14/38] o10  extract  ok    4.6s
[15/38] o10  sql      ok    6.4s
[16/38] o20  classify ok    2.7s
[17/38] o20  extract  ok    5.1s
[18/38] o20  sql      ok   13.5s
[19/38] r10  classify ok    2.7s
[20/38] r10  extract  BAD   8.0s
[21/38] r10  sql      ok    5.2s
[22/38] r10  sql      BAD  17.9s
[23/38] r20  classify ok    2.5s
[24/38] r20  extract  ok    7.4s
[25/38] r20  sql      ok   14.4s
[26/38] b10  classify ok    3.0s
[27/38] b10  extract  ok    2.7s
[28/38] b10  sql      ok    5.2s
[29/38] b10  sql      ok    6.1s
[30/38] b10  sql      BAD   9.4s
[31/38] b10  sql   

## Step 10 — Add LoRA and train
What happens:
1. **Prepare the 4-bit model for training** — turns on gradient checkpointing (recomputes instead of storing, to save memory).
2. **Add LoRA**: small trainable matrices (rank 16) on all seven linear layers of every transformer block. The 8B base weights stay frozen; only about 0.5% of the parameters train.
3. **Train** for 3 epochs, about 47 updates per epoch. The training loss is printed every 5 steps; the **validation loss** after each epoch.
4. A checkpoint is saved to Drive after every epoch, and at the end the **epoch with the lowest validation loss** is kept.

If training loss keeps falling while validation loss rises, the model is memorising — that is why the best epoch, not the last, is kept.

**Resume:** if a checkpoint already exists (after a disconnect), training continues from it. **Re-running this step in the same session:** run Step 7 first, so LoRA is not added twice.

In [1]:
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, set_seed

if isinstance(model, PeftModel):
    raise RuntimeError("LoRA is already attached — run Step 7 again before re-running this step.")

set_seed(SEED)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids":      torch.tensor([b["input_ids"] + [pad] * (width - len(b["input_ids"])) for b in batch]),
        "labels":         torch.tensor([b["labels"] + [-100] * (width - len(b["labels"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (width - len(b["input_ids"])) for b in batch]),
    }

use_bf16 = DTYPE == torch.bfloat16
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        optim="paged_adamw_8bit",
        bf16=use_bf16,
        fp16=not use_bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    ),
    train_dataset=Dataset.from_list(train_data),
    eval_dataset=Dataset.from_list(val_data),
    data_collator=collate,
)

resume = any(CHECKPOINT_DIR.glob("checkpoint-*"))
print("resuming from the last checkpoint" if resume else "starting fresh")
t0 = time.time()
trainer.train(resume_from_checkpoint=True if resume else None)
print(f"\ntraining took {(time.time() - t0) / 60:.1f} min")
for row in trainer.state.log_history:
    if "eval_loss" in row:
        print(f"  epoch {row['epoch']:.0f}: validation loss {row['eval_loss']:.4f}")

NameError: name 'model' is not defined

## Step 11 — Save the adapter to Drive
Saves only the trained LoRA weights (about **175 MB**), not the 16 GB base model. To use the fine-tuned model later you load Qwen3-8B + this adapter (Step 16).

In [ ]:
trainer.save_model(str(ADAPTER_DIR))        # the best epoch, by validation loss
tokenizer.save_pretrained(str(ADAPTER_DIR))
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name:<35} {f.stat().st_size / 1e6:8.1f} MB")

## Step 12 — AFTER training: test the fine-tuned model
The same 38 validation examples, the same scoring — now with the adapter. This is your **"Qwen3 8B + QLoRA"** row.

In [ ]:
tuned_results = run_eval("tuned")

## Step 13 — Compare before and after
Accuracy per task, side by side. With only 11 validation questions, treat these as a first signal, not a final result.

In [ ]:
def accuracy(results, task):
    g = [r for r in results if task == "TOTAL" or r["task"] == task]
    return sum(r["correct"] for r in g) / len(g) if g else float("nan")

print(f"  {'task':<9}{'base':>8}{'tuned':>8}{'change':>9}")
for task in ["classify", "extract", "sql", "TOTAL"]:
    b, t = accuracy(base_results, task), accuracy(tuned_results, task)
    print(f"  {task:<9}{b:>8.0%}{t:>8.0%}{(t - b) * 100:>+8.0f}pp")

## Step 14 — Look at the mistakes
Every validation example the fine-tuned model still gets wrong, with the correct answer underneath. This tells you what the next round of training questions should cover.

In [ ]:
wrong = [r for r in tuned_results if not r["correct"]]
print(f"{len(wrong)} of {len(tuned_results)} wrong\n")
for r in wrong:
    print(f"{r['qid']} · {r['task']}")
    print("  model:  ", r["pred"][:400])
    print("  correct:", r["gold"][:400])
    print()

## Step 15 — Download the adapter and predictions
Packs the trained adapter and both prediction files into one zip and downloads it to your computer (about 160 MB). **Required when `USE_DRIVE = False`** — otherwise everything is lost when the session ends. With Drive it is optional; the files are already in `MyDrive/kqml_finetune/`. Checkpoints are left out: they are large and only needed to resume training.

In [ ]:
import shutil
from google.colab import files

bundle = Path("/content/bundle")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(ADAPTER_DIR, bundle / "adapter")
shutil.copytree(RESULTS_DIR, bundle / "results")
zip_path = shutil.make_archive("/content/qwen3-8b-v1", "zip", bundle)
print(f"{Path(zip_path).name}: {Path(zip_path).stat().st_size / 1e6:.0f} MB")
files.download(zip_path)

## Step 16 — Later: use the saved adapter without retraining
In a **new** Colab session, run Steps 1–8 (they load the base model again), then this cell. It attaches the saved adapter and tests it — no training needed. The adapter must be at `ADAPTER_DIR`: on Drive with `USE_DRIVE = True`, or unzip the Step 15 download there first.

In [ ]:
from peft import PeftModel
model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
reloaded_results = run_eval("tuned_reloaded")